# 07. MCP 에이전트 통합 (MCP Agent Integration)

> **왜 에이전트 통합이 중요한가요?**
>
> MCP 서버를 만드는 것은 도구를 "만드는" 단계예요. 하지만 도구가 아무리 좋아도 에이전트에 연결하지 않으면 쓸모가 없어요. 이 노트북에서는 MCP 도구를 에이전트에 연결해서 **에이전트가 스스로 적절한 도구를 선택하고 사용**하게 만드는 방법을 배워요.

이전 노트북(`06-MCP-Server-Basics.ipynb`)에서는 FastMCP로 서버를 만드는 방법을 배웠어요. 이번에는 그 서버들을 **에이전트에 연결**하는 방법을 살펴볼게요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `MultiServerMCPClient`로 여러 MCP 서버를 동시에 관리할 수 있어요
2. `create_agent`와 MCP 도구를 조합해 대화형 에이전트를 만들 수 있어요
3. `ToolNode` + `StateGraph`로 세밀하게 제어 가능한 MCP 워크플로우를 구성할 수 있어요
4. stdio 서버(날씨)와 HTTP 서버(시간)를 동시에 활용하는 다중 서버 에이전트를 구현할 수 있어요

## 사전 지식

- `06-MCP-Server-Basics.ipynb` — FastMCP 서버 생성, stdio/HTTP 전송 방식
- LangGraph StateGraph, ToolNode, tools_condition
- `create_agent`와 InMemorySaver 사용법
- Python `async`/`await`와 이벤트 루프 기초 — MCP 클라이언트는 비동기예요 (아래 `run_async` 헬퍼로 Jupyter에서 안전하게 호출해요)

## MCP 에이전트 통합 아키텍처

MCP 에이전트 통합은 두 가지 방식으로 구현할 수 있어요.

```mermaid
flowchart TD
    U(["사용자 입력<br>User Input"])
    A1["create_agent<br>고수준 에이전트"]
    A2["StateGraph + ToolNode<br>저수준 커스텀 워크플로우"]
    MC["MultiServerMCPClient<br>다중 서버 관리자"]
    S1["stdio 서버<br>(날씨 정보)"]
    S2["HTTP 서버<br>(현재 시간)"]
    S3["stdio 서버<br>(RAG 검색)"]
    R(["에이전트 응답<br>Agent Response"])

    U --> A1
    U --> A2
    A1 --> MC
    A2 --> MC
    MC --> S1
    MC --> S2
    MC --> S3
    A1 --> R
    A2 --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class U,R input
    class A1,A2 process
    class MC process
    class S1,S2,S3 storage
```

| 방식 | 클래스 | 특징 | 적합한 상황 |
|------|--------|------|------------|
| 고수준 | `create_agent` | 자동 추론-행동 루프 | 빠른 프로토타이핑, 표준 ReAct 패턴 |
| 저수준 | `StateGraph` + `ToolNode` | 노드별 세밀 제어 | 복잡한 분기, 커스텀 로직 |

> 🔑 **핵심 개념**: `MultiServerMCPClient`는 여러 MCP 서버를 하나로 묶어주는 **통합 관리자**예요.

> 💡 **어떤 방식을 선택해야 할까요?**
> - **빠른 프로토타입이 필요하다면** → `create_agent` (코드 5줄로 완성)
> - **조건부 분기, 커스텀 로직이 필요하다면** → `StateGraph + ToolNode` (세밀 제어) 서버마다 다른 전송 방식(stdio, HTTP)을 사용해도 `get_tools()`로 일관되게 도구를 가져올 수 있어요.

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv

load_dotenv()

# LangSmith 추적 설정 (선택사항 - 에이전트 실행 과정을 시각적으로 확인할 수 있어요)
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-MCP-Agent"

## Jupyter에서 비동기 MCP 호출하기: `run_async` 헬퍼

MCP 클라이언트(`MultiServerMCPClient`)는 **비동기(async)** 라서 `await`로 호출해야 해요. 그런데 Jupyter는 이미 자체 이벤트 루프 위에서 돌아가기 때문에, 그 안에서 또 다른 비동기 코드를 직접 실행하면 충돌이 날 수 있어요.

아래 `run_async(coro)` 헬퍼는 **별도 스레드에서 새 이벤트 루프를 만들어** 코루틴을 안전하게 실행해줘요. 동작 원리를 깊이 이해하지 않아도 괜찮아요 — 이 셀을 **그대로 복사해 두고**, 이후 비동기 호출은 `run_async(...)`로 감싸 쓰면 돼요.


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Jupyter 환경에서 비동기 MCP 클라이언트를 안전하게 실행하기 위한 헬퍼예요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 1. MultiServerMCPClient 설정

`MultiServerMCPClient`는 여러 MCP 서버를 동시에 관리하는 클라이언트예요. 각 서버의 도구를 하나의 도구 목록으로 통합해서 에이전트에 전달해줘요.

### 지원하는 전송 방식

| 전송 방식 | 설정 키 | 특징 | 사용 시나리오 |
|----------|---------|------|---------------|
| `stdio` | `command`, `args` | 서버를 서브프로세스로 실행 | 로컬 Python 스크립트 |
| `streamable_http` | `url` | HTTP 엔드포인트로 연결 | 원격 서버, 이미 실행 중인 서버 |

> 💡 **핵심 정리**: `stdio` 방식은 클라이언트가 서버 프로세스를 직접 관리해요. 별도 터미널에서 서버를 켤 필요가 없어서 편리하지만, 서버가 Python 실행 환경에 있어야 해요.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-1. stdio 전송 방식으로 날씨 서버 연결

stdio 방식은 `command`와 `args`로 서버 스크립트를 직접 실행해요. 클라이언트가 서버 프로세스의 수명 주기를 자동으로 관리해줘요.

> ⚠️ **자주 하는 실수**: `args`에서 서버 파일 경로는 **노트북 실행 위치 기준**이에요. 노트북을 다른 디렉토리에서 실행하면 경로 오류가 발생해요. 절대 경로를 사용하거나 `os.path`로 경로를 구성하면 안전해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 1-2. HTTP 전송 방식으로 시간 서버 연결

HTTP 방식은 이미 실행 중인 서버의 URL로 연결해요. 먼저 별도 터미널에서 서버를 실행해야 해요:

```bash
uv run python servers/02_time_server.py
```

> 💡 **실무 팁**: HTTP 방식은 서버를 별도 인프라로 배포할 때 유용해요. 마이크로서비스 아키텍처에서 각 MCP 서버를 독립적으로 배포하고 확장할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 2. create_agent + MCP 도구

`create_agent`는 LangChain V1의 고수준 에이전트 생성 함수예요. LLM과 도구 목록을 전달하면 ReAct 패턴(추론 → 행동 → 관찰 반복)을 자동으로 구현해줘요.

> 🔑 **핵심 개념**: `InMemorySaver`를 `checkpointer`로 전달하면 에이전트가 **대화 상태를 메모리에 유지**해요. 같은 `thread_id`로 여러 번 호출하면 이전 대화를 기억해요.

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 모델: gpt-4o-mini (비용 효율적, 학생 접근성 높음)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 실행 - 스트리밍으로 결과 확인하기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 2-1. 대화 컨텍스트 유지 확인

같은 `thread_id`로 추가 질문을 하면 이전 대화 내용을 기억해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 3. RAG MCP 서버 연결

RAG(Retrieval-Augmented Generation) MCP 서버는 외부 문서에서 관련 정보를 검색하는 기능을 제공해요. MCP를 통해 RAG 기능을 표준화하면 에이전트가 다양한 문서 소스에 접근할 수 있어요.

```mermaid
flowchart LR
    Q(["사용자 질문"])
    AG["에이전트<br>(gpt-4o-mini)"]
    MC["MCP 클라이언트"]
    RS["RAG MCP 서버<br>(stdio)"]
    VDB[("벡터 DB<br>(FAISS)")]
    PDF[("PDF 문서")]
    ANS(["검색 기반 답변"])

    Q --> AG
    AG -->|"retrieve 도구 호출"| MC
    MC --> RS
    RS --> VDB
    VDB --> PDF
    PDF -->|"관련 청크 반환"| VDB
    VDB -->|"검색 결과"| RS
    RS -->|"문서 내용"| MC
    MC -->|"ToolMessage"| AG
    AG --> ANS

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class Q,ANS input
    class AG,MC,RS process
    class VDB,PDF storage
```

> 💡 **실무 팁**: RAG 기능을 MCP 서버로 분리하면 **에이전트 코드와 검색 로직을 분리**할 수 있어요. 나중에 검색 엔진을 FAISS에서 Pinecone으로 바꿀 때 에이전트 코드를 수정할 필요가 없어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 4. 다중 MCP 서버 조합

하나의 에이전트에서 여러 MCP 서버를 동시에 사용할 수 있어요. `MultiServerMCPClient`가 각 서버의 도구를 하나로 합쳐줘요.

> 💡 **핵심 정리**: 이 부분을 라이브로 시연하면서 `get_tools()` 반환값에 **두 서버의 도구가 모두 포함**되는 것을 확인해요. 에이전트는 질문에 따라 적절한 도구를 자동으로 선택해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 5. ToolNode + StateGraph로 커스텀 MCP 워크플로우

`create_agent`는 편리하지만 내부 동작을 제어하기 어려워요. `StateGraph`와 `ToolNode`를 직접 사용하면 **각 노드의 동작을 완전히 제어**할 수 있어요.

```mermaid
flowchart TD
    ST(["START"])
    AG["agent 노드<br>(LLM 추론)"]
    TN["tools 노드<br>(ToolNode - MCP 실행)"]
    EN(["END"])

    ST --> AG
    AG -->|"도구 호출 있음<br>(tools_condition)"| TN
    AG -->|"도구 호출 없음"| EN
    TN -->|"도구 결과 반환"| AG

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class ST,EN input
    class AG,TN process
```

> 🔑 **핵심 개념**: `tools_condition`은 LLM 응답에 도구 호출이 있으면 `"tools"` 노드로, 없으면 `END`로 라우팅하는 조건 함수예요. 이 패턴이 에이전트 루프의 핵심이에요.

In [ ]:
from typing import Annotated, Any

from langchain_core.messages import BaseMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 에이전트 상태 정의 (StateGraph에서 사용할 상태 타입)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
from IPython.display import Image, display

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 그래프 흐름: START → agent → (tools_condition 분기) → tools → agent 루프 또는 END
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 6. 실습: 커스텀 MCP 에이전트 만들기

아래 TODO 블록을 완성하여 날씨와 시간을 **동시에** 물어보고 종합 보고서를 생성하는 에이전트를 만들어보세요.

> ⚠️ **자주 하는 실수**: 다중 질문을 한 번에 처리할 때는 에이전트가 **여러 도구를 순서대로 호출**해요. 이 과정을 스트리밍으로 확인하면서 각 노드가 어떻게 호출되는지 관찰해보세요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 복합 질문 처리 실습
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`MultiServerMCPClient`**: 여러 MCP 서버를 하나로 묶어주는 통합 관리자. `get_tools()`로 모든 서버의 도구를 한 번에 가져와요
- **`setup_mcp_client` 헬퍼**: `server_configs` 딕셔너리로 stdio/HTTP 서버를 유연하게 설정해요
- **`create_agent` 방식**: 빠른 구현에 적합한 고수준 API. ReAct 루프를 자동으로 처리해요
- **`StateGraph` + `ToolNode` 방식**: 세밀한 제어가 필요할 때 사용. 각 노드의 동작을 직접 정의해요
- **`tools_condition`**: 도구 호출 여부에 따라 흐름을 분기하는 조건 함수예요
- **다중 서버 조합**: stdio 서버(로컬)와 HTTP 서버(원격)를 동시에 하나의 에이전트에서 활용할 수 있어요

## 다음 노트북 예고

다음 `08-MCP-Advanced-Servers.ipynb`에서는 **3rd Party MCP 서버 활용**을 배워요. Context7, npx 기반 외부 MCP 서버를 연결하고 복잡한 멀티-서버 워크플로우를 구성하는 방법을 살펴볼게요.